# 18. CLI Reference

Part of the `nuclearff` [User Tutorial](https://nolmacdonald.github.io/nuclearff/tutorial/index.html) notebook series -- every notebook shares one running `./demo` directory and depends on the ones before it having already run. See [README.md](README.md) for the full run order. Run `17_querying_and_provenance.ipynb` first.

Notebooks 1 through 17 covered `nuclearff` as a Python library. Every one of those operations is also available as a single command-line tool, `nuclearff`, for when you want a repeatable, scriptable command instead of a Python session. This notebook is a complete reference for it -- nothing here introduces new functionality; it's the same library, wrapped. Every shell cell below runs `!uv run nuclearff ...` rather than a hardcoded venv path, so it always uses this repo's current source and synced environment. Mirrors the [CLI Reference](https://nolmacdonald.github.io/nuclearff/tutorial/18_cli_reference.html) docs page.

Every command reads a configuration file (`configs/nuclearff.yaml` by default, or whatever `-c`/`--config` points at) and writes beneath `paths.root` (override with `--root`, which is how every example below keeps its output under `./demo` instead of a real project's data).

In [1]:
!uv run nuclearff --help

usage: nuclearff [-h] [--version] [-c CONFIG] [--root ROOT]
                 [--log-level {DEBUG,INFO,WARNING,ERROR,CRITICAL}]
                 <group> ...

League-aware fantasy football research and valuation.

positional arguments:
  <group>
    config              Inspect and create configuration
    sleeper             Read-only Sleeper API access
    ids                 Cross-source player identity resolution
    report              Build draft boards and reports

options:
  -h, --help            show this help message and exit
  --version             show program's version number and exit
  -c, --config CONFIG   Configuration file (default: configs/nuclearff.yaml
                        when present)
  --root ROOT           Override paths.root; every managed directory resolves
                        beneath it
  --log-level {DEBUG,INFO,WARNING,ERROR,CRITICAL}
                        Console log level (default: the configured
                        run.log_level)


## `config` -- project configuration

`init` writes a default configuration file (see `02_configuration.ipynb`'s `default_config`); `--force` overwrites an existing one. `show` prints the fully resolved configuration as YAML. `paths` prints every managed directory and whether it exists; `--ensure` creates any that are missing.

In [2]:
!uv run nuclearff --root ./demo config paths --ensure

ok       /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/raw
ok       /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/processed
ok       /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/cache
ok       /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/manifests
ok       /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts
ok       /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/configs/leagues


In [3]:
!uv run nuclearff --root ./demo config show

model:
  age_curve:
    decline_per_year: 0.025
    min_factor: 0.85
    peak_age: 26
    plateau_end: 28
  baseline: vols
  flex_wr_rate: 0.5
  name: wr_default
  position: WR
  recency:
    weights:
    - 0.5
    - 0.3
    - 0.2
  seasons:
  - 2023
  - 2024
  - 2025
  simulation:
    ceiling_percentile: 0.9
    floor_percentile: 0.1
    games: 17
    n_simulations: 10000
  thresholds:
    min_games: 8
    min_routes: 200
    min_targets: 50
  version: 0.1.0
  weights:
    efficiency: 0.25
    expected_td: 0.1
    situation: 0.15
    volume: 0.5
paths:
  configs: configs
  data: data
  root: demo
run:
  log_level: INFO
  season: 2026
  seed: 2026


## `sleeper` -- read-only Sleeper API access

`fetch-league` is the CLI equivalent of `04_capturing_a_league.ipynb`'s functions, wired together in one call -- every flag below **implies** `--history` (you don't need to pass both), and each writes its own table without disturbing the others:

In [4]:
!uv run nuclearff --root ./demo sleeper fetch-league --league-id 1367225133634191360 --history --standings --matchups --transactions --roster-players --drafts --max-week 3

2026-09-22T03:51:28 [INFO] nuclearff.sleeper.leagues: Wrote 6 Sleeper league(s) (6 with a parsed config) to /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/cache/nuclearff.duckdb


2026-09-22T03:51:29 [INFO] nuclearff.sleeper.standings: Wrote 60 standings, 66 playoff match, and 58 avatar row(s) to /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/cache/nuclearff.duckdb


2026-09-22T03:51:29 [INFO] nuclearff.sleeper.matchups: Wrote 180 Sleeper matchup row(s) to /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/cache/nuclearff.duckdb


2026-09-22T03:51:30 [INFO] nuclearff.sleeper.transactions: Wrote 368 transaction(s) and 546 transaction-player row(s) to /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/cache/nuclearff.duckdb


2026-09-22T03:51:30 [INFO] nuclearff.sleeper.roster_players: Wrote 947 roster-player row(s) to /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/cache/nuclearff.duckdb


2026-09-22T03:51:32 [INFO] nuclearff.sleeper.draft: Wrote 900 draft pick row(s) across 6 league(s) to /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/cache/nuclearff.duckdb
2026-09-22T03:51:32 [INFO] nuclearff.sleeper.snapshot: Wrote Sleeper snapshot to /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/raw/sleeper/1367225133634191360/20260922T105127Z
League:   NUCLEARFF REDRAFT (1367225133634191360)
Season:   2026 status=in_season
Teams:    10
Snapshot: /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/raw/sleeper/1367225133634191360/20260922T105127Z
League config: /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/configs/leagues/1367225133634191360.yaml

Settings to review:
  [WARNING] draft_rounds_mismatch: League settings report draft_rounds=3 but the draft object reports rounds=15.
           -> Trust the draft object. The league field is stale; use 15 rounds for pick-gap and VONA math.
  [WARNING] keepers_in_redraft: League type

`--max-seasons` (default 20) caps how far `--history` walks back; `--max-week` (default 18) caps how many weeks `--matchups`/`--transactions` fetch per season.

`fetch-players` writes the full Sleeper player map to DuckDB; `fetch-projections` fetches one or more weeks of Sleeper's own player projections. `user-leagues`/`user-drafts` resolve a username to their leagues or drafts for a season. `trending` prints the most-added or most-dropped players league-wide.

In [5]:
!uv run nuclearff --root ./demo sleeper fetch-players

2026-09-22T03:51:33 [INFO] nuclearff.sleeper.players: Sleeper player map unchanged (12228 rows); skipped rewriting sleeper_players
Players:  12228
Database: /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/cache/nuclearff.duckdb


In [6]:
!uv run nuclearff --root ./demo sleeper trending --kind add --limit 5

Trending add (last 24h):
  Emanuel Wilson               count=1117242
  Bryce Young                  count=618980
  Tre Tucker                   count=558948
  Tyler Shough                 count=534416
  Adonai Mitchell              count=519096


In [7]:
!uv run nuclearff --root ./demo sleeper user-leagues nolmacdonald --season 2026

User:    nolmacdonald (332632476830679040)
Leagues: 18 for nfl 2026
  1313346760332029952  NUCLEARFF DYNASTY  status=in_season
  1367225133634191360  NUCLEARFF REDRAFT  status=in_season
  1397668648528650240  Antares Sunday Operations  status=in_season
  1387927955862200320  Sacks in the city  status=in_season
  1367169730632220672  NUCLEARFF CHOPPED $50  status=in_season
  1367168783709384704  NUCLEARFF CHOPPED $25  status=in_season
  1367168140898766848  NUCLEARFF CHOPPED OG $10  status=in_season
  1367167628489007104  NUCLEARFF CHOPPED $10 02  status=in_season
  1389346819275771904  Backpack Boyz  status=in_season
  1387966835797798912  Freeman Forever League  status=in_season
  1327454794725457920  Shadows SuperFlex Dynasty  status=in_season
  1313971202405986304  NATO  status=in_season
  1313324299263692800  AB’s Legion of CTE  status=in_season
  1312528581959614464  Kundis   status=in_season
  1312889074398294016  Dynasty Degenerates  status=in_season
  1312142651633451008  Footc

## `ids` -- cross-source identity resolution

Fills missing Sleeper `gsis_id` values from the nflverse `ff_playerids` crosswalk (`04_capturing_a_league.ipynb`). Requires `fetch-players` to have run first.

In [8]:
!uv run nuclearff --root ./demo ids resolve-gsis

2026-09-22T03:51:36 [INFO] nuclearff.nflverse.loader: Fetching nflverse ff_playerids crosswalk


Players:                   12228
gsis_id from Sleeper:      3893
gsis_id from ff_playerids: 3526
Still unresolved:          4809

Skipped 6 ambiguous crosswalk sleeper_id value(s) (maps to more than one player; not used to fill gaps).


## `report` -- draft boards, reports, and visualizations

`auction-board` builds the full auction board (`09_auction_draft_board.ipynb`): CSV, markdown report, and PNG position tables (unless `--no-tables`). Uses the real auction league from that chapter, not the running snake-draft example.

In [9]:
!uv run nuclearff --root ./demo report auction-board 1387966835797798912 --seasons 2024 2025 --as-of-season 2026

2026-09-22T03:51:41 [INFO] nuclearff.pipeline.auction_board: Building auction board for 'Freeman Forever League' (2026): 10 teams, $200 each, 14 roster spots
2026-09-22T03:51:41 [INFO] nuclearff.nflverse.stats: Fetching nflverse seasonal player stats for seasons [2024, 2025]


2026-09-22T03:51:42 [WARNING] nuclearff.scoring.engine: ScoringEngine has no mapping for 27 nonzero scoring key(s); these will not be counted toward fantasy points: blk_kick=2.0, def_pass_def=0.10000000149011612, def_st_fum_rec=2.0, def_st_td=6.0, def_td=6.0, fgm_0_19=3.0, fgm_20_29=3.0, fgm_30_39=3.0, fgm_40_49=3.0, fgm_50p=3.0, fgm_yds_over_30=0.10000000149011612, fgmiss=-1.0, fum_rec=2.0, fum_rec_td=6.0, int=2.0, pts_allow_0=9.0, pts_allow_1_6=6.0, pts_allow_21_27=-1.0, pts_allow_28_34=-2.0, pts_allow_35p=-5.0, pts_allow_7_13=3.0, sack=1.0, safe=2.0, st_td=6.0, tkl_loss=0.25, xpm=1.0, xpmiss=-1.0
2026-09-22T03:51:42 [INFO] nuclearff.pipeline.auction_board: Scoring engine ignored 27 league scoring key(s) with no nflverse stat mapping (kicker/defense/IDP keys are expected here): ['blk_kick', 'def_pass_def', 'def_st_fum_rec', 'def_st_td', 'def_td', 'fgm_0_19', 'fgm_20_29', 'fgm_30_39', 'fgm_40_49', 'fgm_50p']
2026-09-22T03:51:42 [INFO] nuclearff.pipeline.auction_board: Replacement leve

2026-09-22T03:51:43 [INFO] nuclearff.nflverse.loader: Fetching nflverse ff_playerids crosswalk


2026-09-22T03:51:43 [INFO] nuclearff.nflverse.rankings: attach_player_ids: resolved gsis_id for 373 of 404 ranking rows (92.3%)
2026-09-22T03:51:43 [INFO] nuclearff.report.build: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1387966835797798912-2026/auction_board.csv (746 rows)


2026-09-22T03:51:45 [INFO] nuclearff.report.tables: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1387966835797798912-2026/tables/top_12_qb.png (12 QB rows)


2026-09-22T03:51:45 [INFO] nuclearff.report.tables: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1387966835797798912-2026/tables/top_12_rb.png (12 RB rows)


2026-09-22T03:51:45 [INFO] nuclearff.report.tables: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1387966835797798912-2026/tables/top_12_wr.png (12 WR rows)


2026-09-22T03:51:46 [INFO] nuclearff.report.tables: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1387966835797798912-2026/tables/top_12_te.png (12 TE rows)
2026-09-22T03:51:46 [INFO] nuclearff.report.build: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1387966835797798912-2026/report.md
League:          Freeman Forever League (2026)
Budget:          $200/team x 10 teams
Players valued:  746
In draft pool:   140
Report:          /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1387966835797798912-2026/report.md


`playoff-bracket` renders a completed season's winners/losers playoff brackets (`10_draft_and_playoff_visuals.ipynb`). Requires `--standings` to have been run for that league and season first.

In [10]:
!uv run nuclearff --root ./demo report playoff-bracket 1240509989819273216 --season 2025 --league-name "NUCLEARFF REDRAFT"

2026-09-22T03:51:47 [INFO] nuclearff.report.bracket: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1240509989819273216-2025/brackets/winners_bracket.png (7 matches)


2026-09-22T03:51:47 [INFO] nuclearff.report.bracket: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1240509989819273216-2025/brackets/losers_bracket.png (4 matches)
Winners bracket: /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1240509989819273216-2025/brackets/winners_bracket.png
Losers bracket: /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1240509989819273216-2025/brackets/losers_bracket.png


`draft-board` renders a draft as a snake-order grid (`10_draft_and_playoff_visuals.ipynb`). Omitting `--draft-id` uses the league's most recent draft.

In [11]:
!uv run nuclearff --root ./demo report draft-board 1367225133634191360

2026-09-22T03:51:49 [INFO] nuclearff.report.draft_board: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-draft-board/1367225133646778368.png (150 picks)
Picks:       150
Draft board: /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-draft-board/1367225133646778368.png


`trades` renders all ten trade-history visualizations (`14_trade_network.ipynb`); `wins` renders the cumulative-wins step chart (`15_wins_and_leagues.ipynb`); `draft-order` renders a manager's draft-slot history (`10_draft_and_playoff_visuals.ipynb`). All three default to only managers currently rostered in `league_id`'s own season -- `--all-users` includes every manager across the league's full history instead.

In [12]:
!uv run nuclearff --root ./demo report trades 1367225133634191360 --all-users

2026-09-22T03:51:50 [INFO] nuclearff.report.trades: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-trades/trades_by_manager.png (15 managers)


2026-09-22T03:51:51 [INFO] nuclearff.report.trades: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-trades/trades_heatmap.png (15 managers)


2026-09-22T03:51:51 [INFO] nuclearff.report.trades: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-trades/trade_network.png (15 managers, 5 edges)


2026-09-22T03:51:51 [INFO] nuclearff.report.trades: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-trades/trade_leaderboard.png (15 managers)


2026-09-22T03:51:51 [INFO] nuclearff.report.trades: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-trades/manager_pair_leaderboard.png (5 pairs)


...

2026-09-22T03:51:52 [INFO] nuclearff.report.trades: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-trades/manager_season_heatmap.png (15 managers, 6 seasons)


2026-09-22T03:51:52 [INFO] nuclearff.report.trades: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-trades/cumulative_trades.png (5 managers)


2026-09-22T03:51:52 [INFO] nuclearff.report.trades: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-trades/trade_partner_diversity.png (15 managers)


2026-09-22T03:51:52 [INFO] nuclearff.report.trades: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-trades/chord_diagram.png (15 managers)


Managers:                 15
Trades by manager:        /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-trades/trades_by_manager.png
Trades heatmap:           /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-trades/trades_heatmap.png
Trade network:            /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-trades/trade_network.png
Trade leaderboard:        /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-trades/trade_leaderboard.png
Manager-pair leaderboard: /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-trades/manager_pair_leaderboard.png
Manager x season heatmap: /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-trades/manager_season_heatmap.png
Trades over time:         /Users/nmacdonald/projects/sandbox/nuclearff/e

In [13]:
!uv run nuclearff --root ./demo report wins 1367225133634191360 --all-users

2026-09-22T03:51:54 [INFO] nuclearff.report.wins: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-wins.png (15 managers)
Managers: 15
Wins:     /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-wins.png


In [14]:
!uv run nuclearff --root ./demo report draft-order 1367225133634191360 --all-users

2026-09-22T03:51:55 [INFO] nuclearff.report.draft: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-draft-order.png (15 managers)
Managers:    15
Draft order: /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-draft-order.png


`on-this-day` renders transactions matching today's (or a given) calendar-date anniversary, across a league's full history (`13_league_history.ipynb`).

In [15]:
!uv run nuclearff --root ./demo report on-this-day 1367225133634191360

2026-09-22T03:51:57 [INFO] nuclearff.report.on_this_day: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-on-this-day-09-22.png (14 matches)
Date:        September 22
Matches:     14
On this day: /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-on-this-day-09-22.png


`performance`/`season-performance` render actual-vs-projected over/underperformer tables for one week or a full season (`15_wins_and_leagues.ipynb`'s "Actual vs. projected performance"). Both default to starters only -- `--all-players` includes bench players. `season-performance`'s `--min-games` (default 3) keeps a single huge-delta week from dominating a season ranking.

In [16]:
!uv run nuclearff --root ./demo report performance 1367225133634191360 --week 1

2026-09-22T03:51:59 [INFO] nuclearff.report.performance: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-week1-performance.png (10 overachievers, 10 underperformers)
Players:  90
Report:   /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/1367225133634191360-week1-performance.png


`user-leagues` renders a Sleeper user's leagues for a season as a PNG table (`15_wins_and_leagues.ipynb`).

In [17]:
!uv run nuclearff --root ./demo report user-leagues nolmacdonald --season 2026

2026-09-22T03:52:01 [INFO] nuclearff.report.user_leagues: Wrote /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/332632476830679040-leagues/2026.png (18 leagues)


User:    nolmacdonald (332632476830679040)
Leagues: /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts/332632476830679040-leagues/2026.png


> **Why this matters:** every command above is a thin wrapper over exactly the same Python functions the first 17 notebooks called directly -- `report trades` is `load_trades` + `manager_trade_counts` + the ten `render_*` calls from `14_trade_network.ipynb`, nothing more. If a command's output ever surprises you, the fastest way to understand why is to go read (or re-run, cell by cell) the matching chapter's Python instead of guessing from the flag names -- every wiring decision is visible in code there, not hidden behind a flag.

## What's Next

That's every `nuclearff` command, and the close of this notebook series. If a command's behavior isn't clear from its flags, the corresponding earlier notebook walks through the same functionality as plain Python -- often the faster way to understand *why* a command produces what it does.

## See Also

- `01_introduction.ipynb` -- the start of this series.
- [CLI Reference](https://nolmacdonald.github.io/nuclearff/tutorial/18_cli_reference.html) -- the matching docs page.
- [API Reference](https://nolmacdonald.github.io/nuclearff/api/index.html) -- full reference for every public class and function used throughout this tutorial.